# 飯店訂房取消預測：排除資料洩漏與時間外推之評估

人工智慧期中／期末專題的重現 notebook。第 1、2 節為共用設定；第 3–5 節重建期中報告，第 6–8 節為期末（rolling-origin、重新訓練、特徵、指標、評估工具與網頁展示）。只想看期末結果，可略過第 3–5 節。

- 不需要 GPU；CPU 執行約需 30 至 60 分鐘（E1–E3 共數十次訓練）。
- 原始結果於 Windows、Python 3.13 產生；在 Colab 上 LightGBM 與 Random Forest 的數字可能在小數點後第三位有些微差異。

## 1. 下載程式與資料

In [ ]:
!git clone -q https://github.com/qawszx992016910/hotel-cancellation-prediction.git
%cd hotel-cancellation-prediction

## 2. 安裝與報告相同版本的套件

程式以 `!python` 子行程執行，因此不需要重新啟動執行階段。

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq install -y fonts-noto-cjk > /dev/null  # 圖表中文字型

## 3. 主實驗（E1–E4）

In [ ]:
!python -u code/run_experiments.py

## 4. 補充分析（門檻、bootstrap、重複列與時間組成對照）

In [ ]:
!python -u code/run_extra.py
!python -u code/run_overlap.py

## 5. 檢視主要結果

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
for f in ['summary_e1_e2', 'summary_e3', 'e4_temporal', 'e4_bootstrap', 'e2_threshold_equiv', 'val_threshold']:
    print('=' * 20, f)
    print(pd.read_csv(f'results/{f}.csv').to_string())

In [ ]:
from IPython.display import Image, display
for f in ['fig_confusion_lgbm', 'fig_roc', 'fig_feature_importance', 'fig_class_distribution']:
    display(Image(f'results/{f}.png', width=600))

## 6. 期末實驗（步驟 1–4，約 20 分鐘）

1. rolling-origin 時間驗證＋事先定義的選模規則
2. 重新訓練策略（expanding／6 個月／3 個月）
3. 衍生特徵（週末、國定假日、agent 歷史取消率）
4. 補充指標（排除不退款 Recall、每日前 10% Precision）

In [ ]:
!python -u code/final_01_rolling_origin.py
!python -u code/final_02_retraining.py
!python -u code/final_03_features.py
!python -u code/final_04_metrics.py

## 7. 評估工具

輸入與本資料相同欄位結構的 CSV，輸出洩漏檢查、三種切分的效能落差與門檻—Recall 曲線。要評估自己的資料，把 `--data` 換成上傳的檔案路徑。

In [ ]:
!python -u code/booking_eval.py --data data/hotel_bookings.csv --out results/eval_report
from IPython.display import HTML
HTML(open('results/eval_report/report.html', encoding='utf-8').read())

## 8. 網頁展示（每日聯繫名單＋單筆預測）

產生單一 HTML 檔 `demo/index.html`，下載後用瀏覽器開啟即可。

In [ ]:
!python -u code/build_demo.py
from google.colab import files
files.download('demo/index.html')